In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
import joblib
import optuna
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

# Load data from binary classification processed directory
X_train = pd.read_csv('../../dataset_processed/X_train_final.csv')
y_train = pd.read_csv('../../dataset_processed/y_train_final.csv').values.ravel()

neg_count = np.sum(y_train == 0)
pos_count = np.sum(y_train == 1)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"Negative class (0): {neg_count}, Positive class (1): {pos_count}")


## Phase 1: Baseline Model (MLP + SMOTE Pipeline)

In [ ]:
import os
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['f1_macro', 'recall_macro', 'accuracy']

# Fold-isolated SMOTE (30%) + baseline MLP
base_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.30, random_state=42)),
    ('mlp', MLPClassifier(max_iter=300, random_state=42, early_stopping=True, n_iter_no_change=10))
])

cv_results_base = cross_validate(base_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
baseline_f1 = cv_results_base['test_f1_macro'].mean()

print(f"Mean CV Macro F1: {baseline_f1:.4f}")
print(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}")

base_pipeline.fit(X_train, y_train)
y_pred_base = base_pipeline.predict(X_train)
report_base = classification_report(y_train, y_pred_base)
print("\nClassification Report on Full Train Data (Baseline):\n", report_base)

os.makedirs('results', exist_ok=True)
with open('results/baseline_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {baseline_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_base['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_base['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Baseline):\n")
    f.write(report_base)


## Phase 2: Hyperparameter Tuning with Optuna

In [ ]:
import optuna
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

def objective(trial):
    hidden_layer_sizes = trial.suggest_categorical('mlp__hidden_layer_sizes', [
        (100,),
        (64, 32),
        (128, 64),
        (50, 25)
    ])
    activation = trial.suggest_categorical('mlp__activation', ['relu', 'tanh'])
    learning_rate_init = trial.suggest_float('mlp__learning_rate_init', 1e-4, 1e-2, log=True)
    alpha = trial.suggest_float('mlp__alpha', 1e-5, 1e-1, log=True)  # L2 regularization
    
    pipeline = Pipeline([
        ('smote', SMOTE(sampling_strategy=0.30, random_state=42)),
        ('mlp', MLPClassifier(
            max_iter=300,
            random_state=42,
            hidden_layer_sizes=hidden_layer_sizes,
            activation=activation,
            learning_rate_init=learning_rate_init,
            alpha=alpha,
            early_stopping=True,
            n_iter_no_change=10
        ))
    ])
    
    cv_results = cross_validate(pipeline, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    return cv_results['test_score'].mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
print("Starting MLP Neural Network Deep Hyperparameter Optimization (20 trials)...")
study.optimize(objective, n_trials=20, show_progress_bar=False)

print(f"\nBest Params: {study.best_params}")
print(f"Best Macro F1: {study.best_value:.4f}")


## Phase 3: Tuned Eval & Comparison

In [ ]:
import os
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_validate
from sklearn.metrics import classification_report
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

best_params = study.best_params

tuned_pipeline = Pipeline([
    ('smote', SMOTE(sampling_strategy=0.30, random_state=42)),
    ('mlp', MLPClassifier(
        max_iter=400,
        random_state=42,
        hidden_layer_sizes=best_params['mlp__hidden_layer_sizes'],
        activation=best_params['mlp__activation'],
        learning_rate_init=best_params['mlp__learning_rate_init'],
        alpha=best_params['mlp__alpha'],
        early_stopping=True,
        n_iter_no_change=10
    ))
])

cv_results_tuned = cross_validate(tuned_pipeline, X_train, y_train, cv=cv, scoring=scoring, n_jobs=-1)
tuned_f1 = cv_results_tuned['test_f1_macro'].mean()

print(f"Mean CV Macro F1 (Tuned): {tuned_f1:.4f}")
print(f"Mean CV Macro Recall (Tuned): {cv_results_tuned['test_recall_macro'].mean():.4f}")
print(f"Mean CV Accuracy (Tuned): {cv_results_tuned['test_accuracy'].mean():.4f}")

tuned_pipeline.fit(X_train, y_train)
y_pred_tuned = tuned_pipeline.predict(X_train)
report_tuned = classification_report(y_train, y_pred_tuned)
print("\nClassification Report on Full Train Data (Tuned):\n", report_tuned)

os.makedirs('results', exist_ok=True)
with open('results/tuned_metrics.txt', 'w') as f:
    f.write(f"Mean CV Macro F1: {tuned_f1:.4f}\n")
    f.write(f"Mean CV Macro Recall: {cv_results_tuned['test_recall_macro'].mean():.4f}\n")
    f.write(f"Mean CV Accuracy: {cv_results_tuned['test_accuracy'].mean():.4f}\n\n")
    f.write("Classification Report on Full Train Data (Tuned):\n")
    f.write(report_tuned)

print("=== MLP Performance Comparison ===")
print(f"Baseline Macro F1: {baseline_f1:.4f}")
print(f"Tuned Macro F1:    {tuned_f1:.4f}")
print(f"Improvement:       {tuned_f1 - baseline_f1:+.4f}")


## Phase 4: Save Model

In [ ]:
import joblib

if tuned_f1 > baseline_f1:
    print("Tuned model is better. Saving tuned pipeline...")
    final_model = tuned_pipeline
else:
    print("Baseline model is better or equal. Saving baseline pipeline...")
    final_model = base_pipeline

joblib.dump(final_model, 'mlp_best_model.pkl')
print("Optimal MLP Pipeline saved to mlp_best_model.pkl")
